# 07 — Cross-Validation Analysis

## 0. Why not plain k-fold?

Our data is an **autocorrelated trade time-series** with look-back rolling features. Shuffled `KFold` is **invalid**:

- adjacent trades are near-duplicates → random folds leak almost-identical rows into both train and test, **inflating** scores;
- features like `ret_*`, `vol_z_*`, `run_length`, `pct_range_*` are computed from neighbouring rows → splitting mid-window **leaks** across the boundary.

## Setup

In [1]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from src.data import load_trades, load_split, SYMBOLS, DATES
from src.cv import PurgedBlockSplit, leave_one_symbol_out, purged_oof_proba
from src.baselines import tick_rule
from src.evaluate import metrics, compare_classifiers
from src.models import gbm, sequence as seq
from src.models.sequence import SequenceConfig

pd.set_option('display.float_format', '{:.4f}'.format)
print('train+val days used for CV; test day held out for the final scoreboard only')

train+val days used for CV; test day held out for the final scoreboard only


## 1. The splitters
A quick look at what each splitter produces.

In [2]:
# PurgedBlockSplit: contiguous test blocks with an embargo gap removed from train
for tr, te in PurgedBlockSplit(n_splits=4, embargo=5).split(40):
    print(f'test rows {te[0]:>2}-{te[-1]:>2}  | train size {len(tr)} (embargoed around the block)')

print()
frames = {s: pd.concat([load_trades(s, DATES['train']), load_trades(s, DATES['val'])]) for s in SYMBOLS}
for test_sym, train_frames, test_df in leave_one_symbol_out(frames):
    print(f'fold: train on {[len(f) for f in train_frames]} rows, test on {test_sym} ({len(test_df)} rows)')

test rows  0- 9  | train size 25 (embargoed around the block)
test rows 10-19  | train size 20 (embargoed around the block)
test rows 20-29  | train size 20 (embargoed around the block)
test rows 30-39  | train size 25 (embargoed around the block)

fold: train on [63570] rows, test on WIFUSDT (25673 rows)
fold: train on [25673] rows, test on ZAMAUSDT (63570 rows)


## 2. GBM — leave-one-symbol-out CV
Estimate of how the XGBoost generalises to a symbol it never trained on.

In [3]:
gbm_cv = gbm.cross_val_report(verbose=True)
print('\nmean macro-F1 across symbols:',
      f"{np.mean([m['macro_f1'] for m in gbm_cv.values()]):.4f}")

[GBM CV] hold-out WIFUSDT     acc=0.6871  macro_f1=0.6871
[GBM CV] hold-out ZAMAUSDT    acc=0.7483  macro_f1=0.7421

mean macro-F1 across symbols: 0.7146


The cross-symbol numbers (WIF held-out ≈ 0.69, ZAMA held-out ≈ 0.74) are **well below the pooled-val 0.77** — the GBM relies on symbol-specific structure that doesn't transfer.

## 3. GBM hyperparameter tuning

`tune_gbm` runs leave-one-symbol-out CV with **per-fold early stopping**, so the leaderboard reflects the same early-stopped model that would be saved.

In [4]:
best_params, leaderboard = gbm.tune_gbm(save=False, verbose=True)
print()
print(leaderboard.to_string(index=False, float_format='{:.4f}'.format))

[tune] {'max_depth': 3, 'learning_rate': 0.03}  cv_macro_f1=0.6763  (mean best_iter=304)
[tune] {'max_depth': 3, 'learning_rate': 0.1}  cv_macro_f1=0.7200  (mean best_iter=64)
[tune] {'max_depth': 5, 'learning_rate': 0.03}  cv_macro_f1=0.6983  (mean best_iter=220)
[tune] {'max_depth': 5, 'learning_rate': 0.1}  cv_macro_f1=0.6532  (mean best_iter=86)
[tune] {'max_depth': 7, 'learning_rate': 0.05}  cv_macro_f1=0.6895  (mean best_iter=104)

best params: {'max_depth': 3, 'learning_rate': 0.1}  (cv_macro_f1=0.7200)

 max_depth  learning_rate  best_iter_mean  cv_macro_f1
         3         0.1000         63.5000       0.7200
         5         0.0300        220.5000       0.6983
         7         0.0500        104.5000       0.6895
         3         0.0300        303.5000       0.6763
         5         0.1000         86.5000       0.6532


## 4. LSTM — leave-one-symbol-out CV
Train on one symbol, test on the other. (~40 s — trains two LSTMs.)

In [5]:
lstm_cv = seq.cross_val_report(frames, SequenceConfig(epochs=25, patience=5), verbose=True)

[LSTM CV] hold-out WIFUSDT     acc=0.5597  macro_f1=0.4844
[LSTM CV] hold-out ZAMAUSDT    acc=0.7657  macro_f1=0.7650


Single-symbol LSTM does not transfer to WIFUSDT (held-out WIF macro-F1 ≈ 0.48 — near random), while held-out ZAMA is reasonable (≈ 0.77). Trained on ZAMA alone, the LSTM overfits ZAMA's dynamics.